# Current SOXX daily Qlib price-basis seam audit

Read-only existing inventory; official split document is event timing, not price history.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/soxx_forward_daily_basis_seam_audit_20261112';plan=json.loads((card/'plan.json').read_text());source=plan['local_sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for key in ['panel','qlib_manifest','futu_raw']:
 assert sha(root/source[key])==source[key+'_sha256']
manifest=json.loads((root/source['qlib_manifest']).read_text());assert manifest['as_of']=='2026-11-11'
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2026-10-29',end_time='2026-11-05',freq='day'))
q=D.features(['SOXL','SOXX'],['$open','$high','$low','$close','$volume'],start_time='2026-10-29',end_time='2026-11-05',freq='day')
panel=pd.read_csv(root/source['panel']).set_index('date');raw=pd.read_csv(root/source['futu_raw']).set_index('date')
dates=['2026-10-29','2026-10-30','2026-11-04','2026-11-05']
for date in dates:
 for sym in ['SOXL','SOXX']:
  v=float(q.loc[(sym,pd.Timestamp(date)),('$close')]);m=float(panel.loc[date,sym.lower()+'_close'])
  assert abs(v-m)<5e-5
r=raw.loc['2026-10-30'];old=float(panel.loc['2026-10-29','soxx_close']);new=float(panel.loc['2026-10-30','soxx_close'])
vendor_prior=float(r.last_close);vendor_change=float(r.change_rate)/100
naive=new/old-1;vendor_derived=new/vendor_prior-1
assert abs(vendor_derived-vendor_change)<1e-6
assert abs(old/vendor_prior-3)<1e-6
report={'status':'needs_source_basis_repair_before_forward','historical_research_passed':False,
 'source_identity':{k:source[k+'_sha256'] for k in ['panel','qlib_manifest','futu_raw']},
 'qlib_as_of':manifest['as_of'],'calendar_end':manifest['calendar_end'],'checked_dates':dates,
 'split_effective_open_from_official_sec':'2026-11-05','panel_2026_10_29_close':old,'panel_2026_10_30_close':new,
 'vendor_2026_10_30_last_close':vendor_prior,'panel_naive_return':naive,'vendor_reported_change_rate':vendor_change,
 'vendor_price_derived_return':vendor_derived,'old_panel_to_vendor_prior_close_ratio':old/vendor_prior,
 'max_qlib_panel_close_error_lt':5e-5,'new_fits':0,'new_accounts':0,'data_rewrites':0,
 'scope':'existing append-only daily panel mixes SOXX QFQ price vintages; cannot use these new rows for forward SOXX return or native account until a single qualified price basis is rebuilt and split shares/cash are reconciled'}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 R.save_objects(frozen_plan=plan,audit=report,qlib_rows=q,panel_rows=panel.loc[dates,['soxl_close','soxx_close']],futu_row=r.to_dict())
 rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id)
local=card/'qlib_recorder'/eid/rid;names=['frozen_plan','audit','qlib_rows','panel_rows','futu_row']
verification={'recorder_id':rid,'entries':[{'path':(local/'artifacts'/k).relative_to(card).as_posix(),'sha256':sha(local/'artifacts'/k),'bytes':(local/'artifacts'/k).stat().st_size} for k in names]}
(card/'verification_manifest.json').write_text(json.dumps(verification,indent=2)+'\n')
summary={'experiment':plan['experiment'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str((card/'qlib_recorder').resolve()),'recorder_local_dir':str(local.resolve()),'plan_sha256':sha(card/'plan.json'),**report,'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':names}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
check,rr=load_verified_recorder_objects(card,expected_recorder_id=rid,artifacts=['audit','qlib_rows'],expected_manifest_sha256=sha(card/'verification_manifest.json'))
assert check['audit']==report;pd.testing.assert_frame_equal(check['qlib_rows'],q)
(card/'readback_receipt.json').write_text(json.dumps({'recorder_id':rid,'summary_sha256':sha(card/'summary.json'),'manifest_sha256':sha(card/'verification_manifest.json'),'readback':rr},indent=2)+'\n')
print(json.dumps({'status':report['status'],'naive_return':naive,'vendor_return':vendor_change,'basis_ratio':old/vendor_prior,'recorder_id':rid},indent=2))

{
  "status": "needs_source_basis_repair_before_forward",
  "naive_return": -0.6941384816058849,
  "vendor_return": -0.08241544389803429,
  "basis_ratio": 3.000000003006656,
  "recorder_id": "c6060e4393844a11a174fef112931781"
}


## Conclusion

**SOXX forward evidence unavailable from the current shared daily Qlib store.** The 2026-10-29/30 joined close implies −69.41%, while the retained Futu row uses a prior close exactly one third of the saved panel close and reports −8.24%. The official SOXX 3:1 split is recognized at 2026-11-05 open. This is a mixed price-vintage seam. The new ingestion guard rejects the retained row; no data, model, account or paper SIM was changed. See `conclusion.md`, `summary.json`, and readback receipts.